# RSW / 05 — Burst entry on 1-minute bars: with or against the first burst minute (2021-01 – 2023-06)

Protocol: [`RSW 009`](../../discussions/MNQ_Regime_Switching/009_claude_rsw05_burst_entry_preregistration.md). The volatility model and the 2-state regime models (M1 HMM, WB2 Weibull HSMM) are refitted on 1-minute bars. Ticks per contract (1 tick = $0.50).

- **Entry**: at the close of the first 1-minute candle with P(burst) > 0.5, **with** or **against** that candle.
- **Exit**: at the time stop (mean burst duration in minutes + 5 minutes), at a stop, or flat at 15:55 Chicago.
- **Strategies**: 2 models × 2 directions × 5 stops (none, candle extreme + 2 ticks, 1.5 and 3 × 5-minute σ̂, 40 ticks) = 20.
- **Gates**:
  - **G1**: ≥ 150 trades.
  - **G2**: net > 0, Holm across 20 ≤ 0.05.
  - **G3**: net > 0 in ≥ 2 of 3 periods.
  - **G4**: beats the 95th percentile of the random-direction placebo.

In [ ]:
import json, os, sys, tempfile, pickle
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("RSW_SYNTHETIC") == "1"
SYNTH_NULL = os.environ.get("RSW_SYNTH_NULL") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("RSW_SYNTH_DIR") or tempfile.mkdtemp(prefix="rsw_")
    os.environ["RSW_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import mnq_predict as P
from equity_trading import regime_ar as R
from equity_trading import hsmm as HS
from equity_trading import burst_entry as E
from equity_trading import burst_reversion as B
from equity_trading import micro_bars as MB

CFG = P.SYNTH_CFG if SYNTHETIC else P.MRPConfig()
RC, EC = R.RSWConfig(), E.BEConfig()
HC = HS.HSMMConfig(D=EC.hsmm_D)
D = dt.date
OUT = Path("05"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 100)
print("engines", E.BE_VERSION, HS.HSMM_VERSION, R.RSW_VERSION, ("| synthetic " + ("null" if SYNTH_NULL else "planted continuation")) if SYNTHETIC else "| real data")
if SYNTHETIC and not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
    E.write_synthetic_1m(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.train[1], CFG, cont=0.0 if SYNTH_NULL else 0.8)

In [ ]:
if SYNTHETIC:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2020, 7, 1)), CFG.refits_train, CFG.train[1]
    PERIODS = {"2020Q3": (D(2020, 7, 1), D(2020, 9, 30)), "2020Q4": (D(2020, 10, 1), D(2020, 12, 31))}
else:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2021, 1, 1)), CFG.refits_train, CFG.train[1]
    PERIODS = {"2021": (D(2021, 1, 1), D(2021, 12, 31)), "2022": (D(2022, 1, 1), D(2022, 12, 31)),
               "2023H1": (D(2023, 1, 1), D(2023, 6, 30))}
NEED_PERIODS = int(np.ceil(2 / 3 * len(PERIODS)))
m1 = MB.load_1m(CFG.root, CFG.warmup_start - dt.timedelta(days=3), END + dt.timedelta(days=1))
g = E.grid_1m(m1, EC)
g = g[(g["trade_date"] >= CFG.warmup_start) & (g["trade_date"] <= END)]
del m1
cov = E.coverage(g)
cov["flag"] = np.where(cov["minutes_per_day"] < 700, "LOW", "")
print("data coverage by month (minutes per CME day; a full MNQ day has up to 1,380 traded minutes):")
display(cov.round(0).T)
print("months flagged LOW:", list(cov.index[cov["flag"] == "LOW"].astype(str)) or "none")
expected = pd.period_range(pd.Period(CFG.warmup_start, "M"), pd.Period(END, "M"), freq="M")
missing = [str(p) for p in expected if p not in cov.index]
print("months with no data at all:", missing or "none")

In [ ]:
sig2 = E.vol_walk_forward(g, FIRST_FIT, REFITS, END, EC)
S = E.build_series_1m(g, sig2, EC)
S = S[S["trade_date"] >= CFG.train[0]]
td = S["trade_date"].to_numpy()
o = S["obs"].to_numpy(bool)
zq = S.loc[o, "z"].groupby(pd.to_datetime(S.loc[o, "trade_date"]).dt.to_period("Q")).var()
print(f"{len(S):,} minute rows | observed {o.sum():,} | var(z) by quarter:"); display(zq.round(3).to_frame().T)
FLAT = E.flat_index(S, EC)

## 1. Regime fits on 1-minute bars (walk-forward, cached one by one)

In [ ]:
cache = OUT / f"fits_{E.BE_VERSION}{'_synth' if SYNTHETIC else ''}{'_null' if SYNTH_NULL else ''}.pkl"
FITS = pickle.load(open(cache, "rb")) if cache.exists() else {}
for a in REFITS:
    fm = td < a
    if ("M1", str(a)) not in FITS:
        t0 = dt.datetime.now(); FITS[("M1", str(a))] = R.fit(S, 2, False, fm, RC)
        pickle.dump(FITS, open(cache, "wb")); print("M1 before", a, dt.datetime.now() - t0)
    if ("WB2", str(a)) not in FITS:
        t0 = dt.datetime.now(); FITS[("WB2", str(a))] = HS.fit(S, fm, FITS[("M1", str(a))], "WB", False, HC)
        pickle.dump(FITS, open(cache, "wb")); print("WB2 before", a, dt.datetime.now() - t0, "| iter", FITS[("WB2", str(a))].n_iter)
edges = list(REFITS) + [END + dt.timedelta(days=1)]
PB, HOLD, info = {}, {}, []
for name in ("M1", "WB2"):
    pb, hold = np.full(len(S), np.nan), np.zeros(len(S), int)
    for a, b in zip(edges[:-1], edges[1:]):
        f = FITS[(name, str(a))]
        seg = (td >= a) & (td < b)
        burst = int(np.argmax(f.s2))
        if name == "M1":
            p = R.filter_run(S, f)[[f"p{k}" for k in range(2)]].to_numpy()
            mean_b, mean_c = 1 / (1 - f.A[burst, burst]), 1 / (1 - f.A[1 - burst, 1 - burst])
        else:
            p = HS.filter_run(S, f, 2.0)[[f"p{k}" for k in range(2)]].to_numpy()
            mean_b, mean_c = f.mean_duration(burst), f.mean_duration(1 - burst)
        pb[seg] = p[seg, burst]
        hold[seg] = int(round(mean_b)) + EC.extra_bars
        info.append({"model": name, "fit_before": str(a), "burst_sd": float(np.sqrt(f.s2[burst])), "calm_sd": float(np.sqrt(f.s2[1 - burst])),
                     "burst_phi": float(f.phi[burst]), "calm_phi": float(f.phi[1 - burst]), "burst_c": float(f.c[burst]),
                     "mean_burst_min": mean_b, "mean_calm_min": mean_c, "hold_min": int(round(mean_b)) + EC.extra_bars,
                     "share_pb_gt_0.5": float(np.mean(pb[seg] > EC.p_burst))})
    PB[name], HOLD[name] = pb, hold
display(pd.DataFrame(info).round(4))

In [ ]:
EV = {n: E.burst_starts(S, PB[n], HOLD[n], EC) for n in ("M1", "WB2")}
for n, ev in EV.items():
    em = ev.groupby(pd.to_datetime(ev["trade_date"]).dt.to_period("M")).size()
    print(f"--- {n}: {len(ev):,} burst starts | up candles {int((ev.candle_sign > 0).sum()):,}, down {int((ev.candle_sign < 0).sum()):,} | hold minutes {sorted(ev['hold'].unique())}")
    print("burst starts per month:"); display(em.to_frame("starts").T)

## 2. Strategies: 2 models × 2 directions × 5 stops, random-direction placebo

In [ ]:
rows, TR = [], {}
for n, ev in EV.items():
    for direction in EC.directions:
        sides = ev["candle_sign"].to_numpy() * (1 if direction == "with" else -1)
        for kind in EC.stops:
            tr = E.simulate(S, FLAT, ev, kind, sides, EC)
            TR[(n, direction, kind)] = tr
            st = B.stats_for(tr, PERIODS)
            plac = [E.simulate(S, FLAT, ev, kind, np.random.default_rng(seed).choice([-1, 1], len(ev)), EC)["net_ticks"].mean()
                    for seed in range(EC.n_placebo)]
            rows.append({"model": n, "direction": direction, "stop": kind, **st, "skipped_no_room": tr.attrs["skipped"],
                         "placebo_mean": float(np.nanmean(plac)), "placebo_p95": float(np.nanquantile(plac, 0.95))})
            print(n, direction, kind, "| trades", st["trades"], "| net", round(st["net"], 2) if st["trades"] >= 3 else None)
G = pd.DataFrame(rows).set_index(["model", "direction", "stop"])
G["p_holm"] = pd.Series(R.holm({k: v for k, v in G["p"].fillna(1.0).items()}))
G["periods_pos"] = sum((G[f"net_{p}"] > 0).astype(int) for p in PERIODS)
G["G1"] = G["trades"] >= EC.min_trades
G["G2"] = (G["net"] > 0) & (G["p_holm"] <= EC.alpha)
G["G3"] = G["periods_pos"] >= NEED_PERIODS
G["G4"] = G["net"] > G["placebo_p95"]
G["pass"] = G[["G1", "G2", "G3", "G4"]].all(axis=1)
for n in ("M1", "WB2"):
    print(f"=== {n}"); display(G.loc[n].round(3).T)

## 3. Drawdowns: when the money is lost (descriptive)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
for i, n in enumerate(("M1", "WB2")):
    for k, direction in enumerate(EC.directions):
        a_ = ax[i, k]
        for kind in EC.stops:
            tr = TR[(n, direction, kind)]
            if len(tr):
                a_.plot(pd.to_datetime(tr["entry_ts"]), tr["net_ticks"].cumsum(), lw=1, label=kind)
        a_.axhline(0, color="grey", lw=0.6); a_.set_title(f"{n}, {direction} the first burst minute: cumulative net ticks"); a_.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "equity.png", dpi=110); plt.show()
monthly = pd.DataFrame({f"{n}/{d}/{k}": TR[(n, d, k)].groupby(pd.to_datetime(TR[(n, d, k)]["trade_date"]).dt.to_period("M"))["net_ticks"].sum()
                        for n in ("M1", "WB2") for d in EC.directions for k in EC.stops if len(TR[(n, d, k)])})
print("net ticks per month, 1 contract:"); display(monthly.round(0))

## 4. Verdict

In [ ]:
passed = ["/".join(k) for k in G.index[G["pass"]]]
verdict = ("CANDIDATE (2026 pre-registration): " + ", ".join(passed)) if passed else "NO_EDGE"
print(verdict)
summary = {"study": "RSW 05", "engine": E.BE_VERSION, "synthetic": SYNTHETIC, "synthetic_null": SYNTH_NULL,
           "coverage": json.loads(cov.reset_index().astype({"index": str} if "index" in cov.reset_index() else {}).to_json(orient="records", default_handler=str)),
           "fits": info, "starts": {k: int(len(v)) for k, v in EV.items()},
           "results": json.loads(G.reset_index().to_json(orient="records")), "verdict": verdict}
json.dump(summary, open(OUT / "summary.json", "w"), indent=1, default=str)
print("written", OUT / "summary.json")